# 05 — Shortlisting and Recommendations

**Purpose:** Generate actionable creator shortlists for partnerships teams and translate scoring outputs into client-facing recommendations.

This notebook produces:
- Objective-driven shortlists (balanced, awareness, engagement)
- Awareness vs Engagement quadrant chart
- Creator leaderboard
- Risk-flagged creators to investigate before recommending
- Summary recommendations for a partnerships recap

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from ingest_real_data import load_scored_channels, load_benchmarks
from scoring import generate_shortlist
from utils import set_plot_style, COLORS, TIER_ORDER, TIER_COLORS, format_number, format_currency

set_plot_style()

In [ ]:
df = load_scored_channels()
scored = df[df['creator_fit_score'].notna()].copy()
print(f'Loaded {len(df)} channels ({len(scored)} with complete scores)')

## 1. Score Distributions

How the three composite scores are distributed across channels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in zip(
    axes,
    ['creator_fit_score', 'awareness_score', 'engagement_suitability_score'],
    ['Creator Fit Score', 'Awareness Score', 'Engagement Score'],
    [COLORS['primary'], COLORS['secondary'], COLORS['success']]
):
    vals = scored[col].dropna()
    ax.hist(vals, bins=25, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(vals.median(), color=COLORS['danger'], linestyle='--',
               label=f'Median: {vals.median():.1f}')
    ax.set_title(title)
    ax.set_xlabel('Score (0–100)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../dashboard/screenshots/score_distributions.png')
plt.show()

## 2. Awareness vs Engagement Quadrant

The key partnerships visual: which channels are strong on reach, engagement, or both?

In [ ]:
both = scored[scored['awareness_score'].notna() & scored['engagement_suitability_score'].notna()].copy()

fig, ax = plt.subplots(figsize=(11, 8))

tier_color_map = dict(zip(TIER_ORDER, TIER_COLORS))
colors = both['follower_tier'].map(tier_color_map).fillna(COLORS['neutral'])

ax.scatter(
    both['awareness_score'], both['engagement_suitability_score'],
    c=colors, alpha=0.6, s=40, edgecolors='white', linewidth=0.5
)

aw_med = both['awareness_score'].median()
en_med = both['engagement_suitability_score'].median()
ax.axvline(aw_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)
ax.axhline(en_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)

ax.text(88, 78, 'STAR CREATORS\nHigh Reach + Engagement', ha='center', fontsize=9,
        color=COLORS['success'], fontweight='bold')
ax.text(12, 78, 'ENGAGEMENT\nSPECIALISTS', ha='center', fontsize=9,
        color=COLORS['secondary'], fontweight='bold')
ax.text(88, 12, 'AWARENESS\nDRIVERS', ha='center', fontsize=9,
        color=COLORS['primary'], fontweight='bold')
ax.text(12, 12, 'NICHE /\nEMERGING', ha='center', fontsize=9,
        color=COLORS['neutral'], fontweight='bold')

ax.set_xlabel('Awareness Score')
ax.set_ylabel('Engagement Suitability Score')
ax.set_title('Creator Segmentation: Awareness vs Engagement')

handles = [mpatches.Patch(color=c, label=t) for t, c in tier_color_map.items()]
ax.legend(handles=handles, title='Follower Tier', loc='lower right')

plt.tight_layout()
plt.savefig('../dashboard/screenshots/awareness_vs_engagement.png')
plt.show()

## 3. Balanced Shortlist — Top 20 Creators

In [ ]:
balanced = generate_shortlist(scored, objective='balanced', top_n=20)

display_cols = [
    'rank', 'channel_name', 'category', 'follower_tier', 'subscribers',
    'creator_fit_score', 'awareness_score', 'engagement_suitability_score',
    'risk_flag_count', 'recommendation'
]
balanced[display_cols]

## 4. Awareness-Focused Shortlist

In [ ]:
awareness = generate_shortlist(scored, objective='awareness', top_n=15)
awareness[display_cols]

## 5. Engagement-Focused Shortlist

In [ ]:
engagement = generate_shortlist(scored, objective='engagement', top_n=15)
engagement[display_cols]

## 6. Creator Leaderboard — Top 20 by Fit Score

In [ ]:
top20 = scored.nlargest(20, 'creator_fit_score')

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    top20['channel_name'][::-1], top20['creator_fit_score'][::-1],
    color=COLORS['primary'], alpha=0.85
)
for bar, val in zip(bars, top20['creator_fit_score'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=9)

ax.set_xlabel('Creator Fit Score')
ax.set_title('Top 20 Channels by Creator Fit Score')
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig('../dashboard/screenshots/creator_leaderboard.png')
plt.show()

## 7. Scores by Follower Tier

In [ ]:
tier_scores = scored.groupby('follower_tier')[[
    'creator_fit_score', 'awareness_score', 'engagement_suitability_score'
]].mean().reindex(TIER_ORDER).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
tier_scores.plot(kind='bar', ax=ax,
    color=[COLORS['primary'], COLORS['secondary'], COLORS['success']])
ax.set_title('Average Scores by Follower Tier')
ax.set_ylabel('Score (0–100)')
ax.legend(['Fit Score', 'Awareness', 'Engagement'], title='Score Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../dashboard/screenshots/scores_by_tier.png')
plt.show()

tier_scores

## 8. Risk-Flagged Channels

In [ ]:
high_risk = scored[scored['risk_flag_count'] >= 3][[
    'channel_name', 'category', 'followers' if 'followers' in scored.columns else 'subscribers',
    'risk_flag_count', 'flag_missing_30d_data', 'flag_low_upload_volume',
    'flag_negative_growth', 'flag_very_high_uploads'
]].sort_values('risk_flag_count', ascending=False).head(15)

print(f'{len(scored[scored["risk_flag_count"] >= 3])} channels have 3+ risk flags')
print('These should be investigated before including in any shortlist.\n')
high_risk

## 9. Benchmark Context

In [ ]:
benchmarks = load_benchmarks()
print('Public Campaign Benchmarks (from Humanz / Ubiquitous case studies):')
print('=' * 70)
for _, row in benchmarks.iterrows():
    parts = [f"{row['campaign']} ({row['brand']})"]
    if pd.notna(row['views']):
        parts.append(f"Views: {format_number(row['views'])}")
    if pd.notna(row['cpm_usd']):
        parts.append(f"CPM: ${row['cpm_usd']:.2f}")
    if pd.notna(row['engagements']):
        parts.append(f"Engagements: {format_number(row['engagements'])}")
    if pd.notna(row['conversions_new_users']):
        parts.append(f"Conversions: {format_number(row['conversions_new_users'])}")
    print(f"  {' | '.join(parts)}")
    print(f"    Source: {row['source_url']}")
    if pd.notna(row['notes']) and row['notes']:
        print(f"    Note: {row['notes']}")
    print()

---

## Summary: Recommendations for Partnerships Teams

Based on the scoring framework applied to 990 real YouTube channels:

**1. Use objective-driven shortlists, not one-size-fits-all rankings.**
The awareness shortlist favors high-subscriber channels with strong 30-day views momentum. The engagement shortlist surfaces channels with high views-per-subscriber ratios and consistent posting. Different campaign goals require different creator profiles.

**2. The quadrant chart is the most useful partnerships visual.**
Star Creators (top-right) score high on both reach and engagement — they are rare and valuable. Most channels specialize in one dimension. Understanding where a creator sits on this chart helps set campaign expectations.

**3. Risk flags are essential before recommending.**
Channels with missing 30-day data, negative subscriber growth, or very high upload volumes (possible network/compilation channels) should be flagged for manual review. The scoring framework filters these out of shortlists, but partnerships teams should know why.

**4. Public campaign benchmarks provide pricing and performance context.**
Real CPMs from Ubiquitous case studies range from $1.47–$11.00. Engagement results from Humanz campaigns show that 200K+ engagements and 6x industry-standard rates are achievable with the right creator-brand fit. These benchmarks help set client expectations.

**5. This analysis is a starting point, not a final recommendation.**
Channel-level data provides directional guidance. A production system would add video-level metrics, audience demographics, brand safety checks, and historical campaign performance to refine shortlists.